In [ ]:
file_path=""


"""parameter"""

In [ ]:
import traceback
from pyspark.sql.functions import (
    col, year, month, dayofmonth, to_timestamp,
    lit, when, length, pow
)
from pyspark.sql.types import IntegerType, StringType, DecimalType, BooleanType
 
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")
 
table = file_path.split("/silver/")[1].split("/")[0]
print(f"Processing silver table: {table}")
 
server   = ""
database = ""
user     = ""
password = ""
url = (
    f"jdbc:sqlserver://{server}:1433;"
    f"database={database};"
    f"encrypt=true;"
    f"trustServerCertificate=false;"
    f"hostNameInCertificate=*.sql.azuresynapse.net;"
    f"loginTimeout=30;"
    f"user={user};"
    f"password={password}"
)
 
ADLS_ROOT   = f"abfss://{}@{}.dfs.core.windows.net"
SILVER_ROOT = f"{ADLS_ROOT}/silver"

In [ ]:
def read_silver(table_name: str):
    result = spark.sql(f"""
        SELECT file_path FROM control.latest_tables
        WHERE is_latest = 'true' AND table_name = '{table_name}'
    """)
    if result.count() == 0:
        raise ValueError(f"[latest_tables] no latest entry found for '{table_name}'")
    path = result.first()["file_path"]
    print(f"  Loading lookup '{table_name}' from: {path}")
    return spark.read.parquet(path)

In [ ]:
table = file_path.split("/silver/")[1].split("/")[0]
print(f"Processing silver table: {table}")

In [ ]:
from pyspark.sql.functions import col, date_format

def todatekey(column):
    if isinstance(column, str):
        column = col(column)
    return date_format(column, "yyyyMMdd").cast("int")

In [ ]:
if table == "department":
    gold_df = df.select(
        col("id")               .cast(IntegerType()).alias("department_key"),
        col("name")             .cast(StringType()),
        col("head_prof_user_id").cast(IntegerType())
    )
    gold_table = "dim_department"
 
elif table == "academic_semester":
    gold_df = df.select(
        col("id")                  .cast(IntegerType()).alias("academic_semester_key"),
        col("name")                .cast(StringType()),
        col("start_date_key") .cast(IntegerType()),
        col("end_date_key")   .cast(IntegerType())
    )
    gold_table = "dim_semester"
 
elif table == "quizzes":
    gold_df = df.select(
        (lit(1) * pow(lit(10), length(col("id").cast(StringType()))) + col("id").cast(IntegerType())).cast(IntegerType()).alias("assessment_key"),
        lit("quiz")                         .cast(StringType()).alias("source_table"),
        col("quiz_type")                    .cast(StringType()).alias("assessment_type"),
        col("title")                        .cast(StringType()),
        col("description")                  .cast(StringType()),
        col("scheduled_date_key")         .cast(IntegerType()),
        col("created_at_key")             .cast(IntegerType()),
        col("duration_min")                 .cast(IntegerType()),
        col("total_marks")                  .cast(IntegerType()),
        lit(None)                           .cast(StringType()).alias("room")
    )
    gold_table = "dim_assessment"
 
elif table == "exams":
    gold_df = df.select(
        (lit(2) * pow(lit(10), length(col("id").cast(StringType()))) + col("id").cast(IntegerType())).cast(IntegerType()).alias("assessment_key"),   
        lit("exam")                 .cast(StringType()).alias("source_table"),      
        col("exam_type")            .cast(StringType()).alias("assessment_type"),  
        lit("Exam")                 .cast(StringType()).alias("title"),             
        lit(None)                   .cast(StringType()).alias("description"),       
        todatekey("exam_date")      .alias("scheduled_date_key"),                  
        lit(None)                   .cast(IntegerType()).alias("created_at_key"),  
        col("duration_min")         .cast(IntegerType()),                           
        col("total_marks")          .cast(IntegerType()),                           
        col("room")                 .cast(StringType())                             
    )
    gold_table = "dim_assessment"
 
elif table == "tickets":
    gold_df = df.select(
        col("id")                  .cast(IntegerType()).alias("ticket_key"),
        col("title")               .cast(StringType()),
        col("description")         .cast(StringType()),
        col("created_by_user_id")  .cast(IntegerType()).alias("student_key"),
        col("assigned_to_user_id") .cast(IntegerType()),
        col("status")              .cast(StringType()),
        col("created_at_key") .cast(IntegerType()).alias("created_at_key"),
        col("updated_at_key") .cast(IntegerType()).alias("updated_at_key")
    )
    gold_table = "fact_ticket"
 
 
elif table == "professors":
    dept = read_silver("department").select(
        col("id")  .alias("dept_id"),
        col("name").alias("department_name")
    )
    gold_df = (
        df.join(dept, df.department_id == dept.dept_id, how="left")
          .select(
              col("user_id")        .cast(IntegerType()).alias("professor_key"),
              col("full_name")      .cast(StringType()),
              col("gender")         .cast(StringType()),
              col("hire_date_key").cast(IntegerType()),
              col("department_id")  .cast(IntegerType()),
              col("department_name").cast(StringType())
          )
    )
    gold_table = "dim_professor"
 
elif table == "courses":
    dept = read_silver("department").select(
        col("id")  .alias("dept_id"),
        col("name").alias("department_name")
    )
    gold_df = (
        df.join(dept, df.department_id == dept.dept_id, how="left")
          .select(
              col("id")            .cast(IntegerType()).alias("course_key"),
              col("code")          .cast(StringType()),
              col("name")          .cast(StringType()),
              col("description")   .cast(StringType()),
              col("credits")       .cast(IntegerType()),
              col("department_id") .cast(IntegerType()),
              col("department_name").cast(StringType())
          )
    )
    gold_table = "dim_course"
 
elif table == "students":
    dept = read_silver("department").select(
        col("id")  .alias("dept_id"),
        col("name").alias("department_name")
    )
    gold_df = (
        df.join(dept, df.department_id == dept.dept_id, how="left")
          .select(
              col("user_id")               .cast(IntegerType()).alias("student_key"),
              col("full_name")             .cast(StringType()),
              col("gender")               .cast(StringType()),
              col("birth_date_key").cast(IntegerType()),
              col("enroll_date_key")  .cast(IntegerType()),
              col("expected_graduation_key") .cast(IntegerType()),
              col("academic_year")         .cast(StringType()),
              col("current_gpa")           .cast(DecimalType(3,2)),
              col("passed_credits")        .cast(IntegerType()),
              col("registered_credits")    .cast(IntegerType()),
              col("department_id")         .cast(IntegerType()),
              col("department_name")       .cast(StringType())
          )
    )
    gold_table = "dim_student"

 
elif table == "course_offerings":
    courses = read_silver("courses").select(
        col("id")           .alias("crs_id"),
        col("code")         .alias("course_code"),
        col("name")         .alias("course_name"),
        col("credits")      .alias("course_credits"),
        col("department_id") .alias("crs_dept_id")
    )
    semesters = read_silver("academic_semester").select(
        col("id")  .alias("sem_id"),
        col("name").alias("semester_name")
    )
    professors = read_silver("professors").select(
        col("user_id")  .alias("prof_user_id"),
        col("full_name").alias("professor_name")
    )
    dept = read_silver("department").select(
        col("id")  .alias("dept_id"),
        col("name").alias("department_name")
    )
    gold_df = (
        df
        .join(courses,    df.course_id          == courses.crs_id,       how="left")
        .join(semesters,  df.semester_id         == semesters.sem_id,     how="left")
        .join(professors, df.professor_user_id   == professors.prof_user_id, how="left")
        .join(dept,       col("crs_dept_id")     == dept.dept_id,         how="left")
        .select(
            col("id")               .cast(IntegerType()).alias("course_offering_key"),
            col("start_date_key").cast(IntegerType()),
            col("end_date_key") .cast(IntegerType()),

            col("course_id")         .cast(IntegerType()),
            col("course_code")       .cast(StringType()),
            col("course_name")       .cast(StringType()),
            col("course_credits")    .cast(IntegerType()),

            col("semester_id")       .cast(IntegerType()),
            col("semester_name")     .cast(StringType()),

            col("professor_user_id") .cast(IntegerType()).alias("professor_id"),
            col("professor_name")    .cast(StringType()),

            col("crs_dept_id")       .cast(IntegerType()).alias("department_id"),
            col("department_name")   .cast(StringType())
        )
    )
    gold_table = "dim_course_offering"
 

 
elif table == "attendance":
    offerings = read_silver("course_offerings").select(
        col("id")              .alias("co_id"),
        col("course_id")       .cast(IntegerType()).alias("co_course_id"),
        col("semester_id")     .cast(IntegerType()).alias("co_semester_id"),
        col("professor_user_id").cast(IntegerType()).alias("co_prof_id"),
    )
 
    status_flag = (
        when(col("status") == "present", lit(1.00))
       .when(col("status") == "late",    lit(0.50))
       .otherwise(lit(0.00))
    )
 
    gold_df = (
        df
        .join(offerings, df.course_offering_id == offerings.co_id, how="left")
        .select(
            col("id")               .cast(IntegerType()).alias("attendance_key"),
            col("student_user_id")  .cast(IntegerType()).alias("student_key"),
            col("course_offering_id").cast(IntegerType()).alias("offering_key"),
            col("co_semester_id")   .alias("semester_key"),
            col("co_course_id")     .alias("course_key"),
            col("co_prof_id")       .alias("professor_key"),
            col("session_date_key").cast(IntegerType()),
            col("recorded_at")        .cast(IntegerType()).alias("recorded_at_key"),
            col("status")           .cast(StringType()),
            status_flag             .cast(DecimalType(3,2)).alias("status_flag")
        )
    )
    gold_table = "fact_attendance"
 
 
elif table == "quiz_submission":
    quizzes = read_silver("quizzes").select(
        col("id")               .alias("q_id"),
        col("course_offering_id").alias("q_offering_id")
    )
    offerings = read_silver("course_offerings").select(
        col("id")              .alias("co_id"),
        col("course_id")       .cast(IntegerType()).alias("co_course_id"),
        col("semester_id")     .cast(IntegerType()).alias("co_semester_id"),
        col("professor_user_id").cast(IntegerType()).alias("co_prof_id"),
    )
    gold_df = (
        df
        .join(quizzes,   df.quiz_id            == quizzes.q_id,          how="left")
        .join(offerings, col("q_offering_id")  == offerings.co_id,       how="left")
        .select(
            (lit(1) * pow(lit(10), length(col("id").cast(StringType()))) + col("quiz_id").cast(IntegerType())).cast(IntegerType()).alias("assessment_key"),
            col("student_user_id")  .cast(IntegerType()).alias("student_key"),
            col("quiz_id")          .cast(IntegerType()).alias("source_key"),
            col("q_offering_id")    .cast(IntegerType()).alias("offering_key"),
            col("co_semester_id")   .alias("semester_key"),
            col("co_course_id")     .alias("course_key"),
            col("co_prof_id")       .alias("professor_key"),
            col("submitted_at_key").cast(IntegerType()),
            col("score")            .cast(DecimalType(10,2)),
            col("percentage")       .cast(DecimalType(5,2))
        )
    )
    gold_table = "fact_assessment_result"
 
elif table == "exam_results":
    exams = read_silver("exams").select(
        col("id")               .alias("ex_id"),
        col("course_offering_id").alias("ex_offering_id")
    )
    offerings = read_silver("course_offerings").select(
        col("id")              .alias("co_id"),
        col("course_id")       .cast(IntegerType()).alias("co_course_id"),
        col("semester_id")     .cast(IntegerType()).alias("co_semester_id"),
        col("professor_user_id").cast(IntegerType()).alias("co_prof_id"),
    )
    gold_df = (
        df
        .join(exams,     df.exam_id            == exams.ex_id,           how="left")
        .join(offerings, col("ex_offering_id") == offerings.co_id,       how="left")
        .select(
            (lit(2) * pow(lit(10), length(col("exam_id").cast(StringType()))) + col("id").cast(IntegerType())).cast(IntegerType()).alias("assessment_key"),
            col("student_user_id")  .cast(IntegerType()).alias("student_key"),
            col("exam_id")          .cast(IntegerType()).alias("source_key"),
            col("ex_offering_id")   .cast(IntegerType()).alias("offering_key"),
            col("co_semester_id")   .alias("semester_key"),
            col("co_course_id")     .alias("course_key"),
            col("co_prof_id")       .alias("professor_key"),
            lit(None)               .cast(IntegerType()).alias("submitted_at_key"),
            col("score")            .cast(DecimalType(10,2)),
            col("percentage")       .cast(DecimalType(5,2))
        )
    )
    gold_table = "fact_assessment_result"
 
else:
    raise ValueError(f"No transformation defined for table: {table}")

In [ ]:
safe_path = file_path.replace("'", "\\'")

try:
    staging_path = f"abfss://@.dfs.core.windows.net/tmp/{gold_table}/"
    gold_df.write.mode("overwrite").parquet(staging_path)
    print(f"Staged {gold_table} to ADLS")


    if gold_table == "fact_assessment_result":
  
        column_list = "(source_id, student_key, assessment_key, offering_key, semester_key, course_key, professor_key, submitted_at_key, score, percentage)"
        copy_sql = f"""
        COPY INTO dbo.{gold_table} {column_list}
        FROM 'https://.dfs.core.windows.net//tmp/{gold_table}/'
        WITH (FILE_TYPE = 'PARQUET', CREDENTIAL = (IDENTITY = 'Managed Identity'))
        """
    else:
        copy_sql = f"""
        COPY INTO dbo.{gold_table}
        FROM 'https://.dfs.core.windows.net//tmp/{gold_table}/'
        WITH (FILE_TYPE = 'PARQUET', CREDENTIAL = (IDENTITY = 'Managed Identity'))
        """

    conn = spark._sc._jvm.java.sql.DriverManager.getConnection(url)
    conn.createStatement().execute(copy_sql)
    conn.close()
    print(f"{gold_table} written to warehouse successfully")

except Exception as e:
    spark.sql(f"""
        UPDATE control.silver_pending
        SET status = 'pending'
        WHERE file_path = '{safe_path}'
    """)
    print(f"Failed to process: {file_path}")
    raise e